# IDK-1 — Inference

Load checkpoint dan generate teks.

**Kaggle input datasets yang perlu ditambahkan:**
- `ripkii/idk1-tokenizer`
- `raisasyahna/idk-1-checkpoint37500` (atau checkpoint lebih baru kalau udah ke-upload)

**GPU:** T4 x1 cukup untuk inference.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import os
import glob
from dataclasses import dataclass
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Config + Architecture

Harus identik dengan training config.

In [ ]:
@dataclass
class IDK1Config:
    vocab_size  : int   = 40_000
    dim         : int   = 768
    n_layers    : int   = 12
    n_heads     : int   = 12
    n_kv_heads  : int   = 4
    ffn_dim     : int   = 2048
    max_seq_len : int   = 1024
    norm_eps    : float = 1e-5
    rope_theta  : float = 500_000.0
    logit_cap   : float = 30.0

    @property
    def head_dim(self):
        return self.dim // self.n_heads


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


def precompute_rope(head_dim, seq_len, theta, device):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t     = torch.arange(seq_len, device=device)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    B, T, H, D = x.shape
    x1  = x[..., :D//2]
    x2  = x[..., D//2:]
    cos = cos[:T].unsqueeze(0).unsqueeze(2)
    sin = sin[:T].unsqueeze(0).unsqueeze(2)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class GroupedQueryAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads    = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim   = cfg.head_dim
        self.groups     = cfg.n_heads // cfg.n_kv_heads
        self.wq  = nn.Linear(cfg.dim, cfg.n_heads    * cfg.head_dim, bias=False)
        self.wk  = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wv  = nn.Linear(cfg.dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.wo  = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.dim,    bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads,    self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.groups, dim=2)
        v = v.repeat_interleave(self.groups, dim=2)
        out = F.scaled_dot_product_attention(
            q.transpose(1,2), k.transpose(1,2), v.transpose(1,2), is_causal=True
        )
        return self.wo(out.transpose(1,2).contiguous().view(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)
        self.up   = nn.Linear(cfg.dim, cfg.ffn_dim, bias=False)
        self.down = nn.Linear(cfg.ffn_dim, cfg.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.dim, cfg.norm_eps)
        self.attn      = GroupedQueryAttention(cfg)
        self.ffn_norm  = RMSNorm(cfg.dim, cfg.norm_eps)
        self.ffn       = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class IDK1Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.embed   = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.layers  = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.dim, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta, device='cpu')
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)

    def forward(self, idx):
        B, T = idx.shape
        x    = self.embed(idx)
        cos  = self.rope_cos[:T]
        sin  = self.rope_sin[:T]
        for layer in self.layers:
            x = layer(x, cos, sin)
        logits = self.lm_head(self.norm(x))
        logits = self.cfg.logit_cap * torch.tanh(logits / self.cfg.logit_cap)
        return logits


print('Architecture OK')

## 2. Load Tokenizer + Checkpoint

In [ ]:
# ── Tokenizer ─────────────────────────────────────────────────────────
TOKENIZER_PATH = '/kaggle/input/idk1-tokenizer/tokenizer.json'
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
print(f'Tokenizer loaded: vocab_size={tokenizer.get_vocab_size()}')

# ── Checkpoint ────────────────────────────────────────────────────────
# Ganti path sesuai dataset yang lu tambahkan:
# - step 37500: /kaggle/input/idk-1-checkpoint37500/checkpoints/latest.pt
# - step terbaru: /kaggle/input/<dataset-name>/checkpoints/latest.pt
CKPT_PATH = '/kaggle/input/idk-1-checkpoint37500/checkpoints/latest.pt'

cfg   = IDK1Config()
model = IDK1Model(cfg).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)

state_dict = ckpt['model']
# Strip DataParallel prefix kalau ada
if any(k.startswith('module.') for k in state_dict):
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()

step_loaded = ckpt.get('step', '?')
val_loss    = ckpt.get('best_val_loss', '?')
params      = sum(p.numel() for p in model.parameters())

print(f'Checkpoint step : {step_loaded}')
print(f'Best val loss   : {val_loss:.4f}')
print(f'Params          : {params/1e6:.2f}M')
print('Model ready!')

## 3. Generate

In [ ]:
@torch.no_grad()
def generate(
    prompt: str,
    max_new_tokens: int = 200,
    temperature: float  = 0.8,
    top_k: int          = 50,
    rep_penalty: float  = 1.3,
) -> str:
    ids = tokenizer.encode(prompt).ids
    x   = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        # Potong kalau context terlalu panjang
        x_in = x[:, -cfg.max_seq_len:]

        logits = model(x_in)[0, -1, :]  # logits token terakhir

        # Repetition penalty
        if rep_penalty != 1.0:
            for tok in set(x[0].tolist()):
                logits[tok] /= rep_penalty

        # Temperature
        logits = logits / temperature

        # Top-k sampling
        if top_k > 0:
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[-1]] = float('-inf')

        probs   = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, 1)
        x       = torch.cat([x, next_id.unsqueeze(0)], dim=1)

    all_ids = x[0].tolist()
    return tokenizer.decode(all_ids)


print('generate() ready')

## 4. Run!

In [ ]:
prompts = [
    'Indonesia adalah negara',
    'Pada tahun 2024, perkembangan kecerdasan buatan',
    'Bahasa Indonesia merupakan bahasa persatuan yang',
    'Presiden Republik Indonesia',
    'Teknologi machine learning saat ini digunakan untuk',
]

for i, prompt in enumerate(prompts):
    print(f'\n{'='*60}')
    print(f'[{i+1}] PROMPT: {prompt}')
    print('-'*60)
    out = generate(prompt, max_new_tokens=150, temperature=0.8, top_k=50, rep_penalty=1.3)
    print(out)

print(f'\n{'='*60}')
print(f'Step {step_loaded} / 100,000 — {step_loaded/1000:.1f}% selesai')

## 5. Interactive (opsional)

Ganti prompt sesuka lu.

In [ ]:
# Ganti teks di sini:
custom_prompt = 'Kopi Indonesia terkenal di dunia karena'

print(f'PROMPT: {custom_prompt}')
print('-'*60)
print(generate(
    custom_prompt,
    max_new_tokens=200,
    temperature=0.9,   # lebih tinggi = lebih random/kreatif
    top_k=40,
    rep_penalty=1.2,
))